# Bareq · 1. Data

Builds the train / val / test data once, with fixed seeds. Splits happen before any cropping, LOL v1/v2 near-duplicates are removed from training, and SIDD is split by scene.

**Settings (right panel):** Accelerator **None**, Internet **On**.
**Inputs:** `ohmahler91/lolv1-and-lolv2`, `rajat95gupta/smartphone-image-denoising-dataset`, `soumikrakshit/div2k-high-resolution-images`.

Output: `/kaggle/working/bareq-data` (about 11 GB). Runtime: roughly 30–60 minutes.

In [ ]:
REPO = "https://github.com/yzdvai/bareq.git"
REF = "main"  # branch, tag or commit

!df -h /tmp /kaggle/working | tail -n 2
!rm -rf /tmp/bareq && git clone -q {REPO} /tmp/bareq && cd /tmp/bareq && git checkout -q {REF} && git log -1 --oneline

In [ ]:
!cd /tmp/bareq && BAREQ_COMMIT=$(git rev-parse HEAD) python -m bareq.prepare --input /kaggle/input --out /kaggle/working/bareq-data

In [ ]:
# Summary to send back
import json
from IPython.display import Image, display
m = json.load(open("/kaggle/working/bareq-data/manifest.json"))
d = m["lol"]["dedupe"]
print("commit:", m["git_commit"][:7], " seconds:", m["seconds"])
print("LOL train/val/test:", len(m["lol"]["train"]), len(m["lol"]["val"]), len(m["lol"]["test"]))
print("LOL dropped:", len(d["dropped"]), "| v1/v2 near-duplicate pairs:", d["v1_v2_near_duplicate_pairs"])
print("LOL test -> nearest non-test distance:", d["test_nearest_nontest_distance"])
print("LOL nearest-distance histogram (0..2, 20 bins):", d["all_nearest_distance_hist"])
print("SIDD:", {k: (v["scenes"], len(v["instances"]), v["files"]) for k, v in m["sidd"].items() if k != "test_crops"})
print("DIV2K train/val/test:", len(m["div2k"]["train"]), len(m["div2k"]["val"]), len(m["div2k"]["test"]), "| patches:", m["div2k"]["train_patches"])
print("leak check:", m["leak_check"])
display(Image("/kaggle/working/bareq-data/preview.jpg"))